# Build a FAIR Chef repository catalog from the re3data API

This notebook builds a structured FAIR Chef **repository catalog** using metadata pulled from the **re3data API**.  
In this example, the first repository added to the catalog is **OpenNeuro**.

In addition to preserving the original repository metadata, the schema now includes **five extra raw-friendly fields** to support future compiler-based filtering across catalogs:

- `domains`
- `data_types`
- `access_profile`
- `supported_funders`
- `human_subjects_support`

These fields are designed to stay close to the source metadata rather than making early decisions or classifications. They provide a lightweight, compiler-ready layer that can be interpreted later without losing provenance or detail. This matches the updated schema you are now using. :contentReference[oaicite:0]{index=0}

## Goal
The goal is to:
1. define a JSON schema for FAIR Chef repository catalog entries,
2. save that schema for reuse,
3. create an empty repository catalog container,
4. pull a repository record from re3data,
5. parse the XML response,
6. map the XML fields into the FAIR Chef schema,
7. populate five additional raw-friendly fields for future compiler use,
8. validate the final normalized repository record,
9. append the validated record to the repository catalog,
10. save the repository catalog as JSON.

## Why the 5 new fields were added
The five new fields were added to make repository records easier to filter and compare in a later compiler stage, while still keeping the current phase faithful to the source.

### 1. `domains`
This stores subject/domain evidence in a structured way, based directly on the repository’s subject metadata.  
It helps preserve domain coverage without forcing early normalization or interpretation.

### 2. `data_types`
This stores the repository’s raw content/data type information in a compiler-friendly field.  
It makes it easier later to filter repositories by the kinds of data they support, such as images, text, raw data, or archived data.

### 3. `access_profile`
This stores access-related evidence in a simplified structured block based on the raw access fields.  
It gives the compiler a clean place to inspect open, restricted, embargoed, or upload-restriction information later.

### 4. `supported_funders`
This stores raw funder names derived from funding-related institutions in the source metadata.  
It supports later matching of repositories to funding agencies without relying on free-text parsing at query time.

### 5. `human_subjects_support`
This stores only raw evidence related to possible human-subject relevance, such as subject values, description text, and policy names.  
It does **not** make a final yes/no decision at this stage, which keeps the record source-aligned and avoids premature assumptions.

## Input source
- Repository example: **OpenNeuro**
- re3data ID: `r3d100010924`
- API endpoint: `https://www.re3data.org/api/v1/repository/r3d100010924`

## Output files
- `repository_schema.json` — the FAIR Chef schema for repository catalog entries
- `repository_catalog.json` — the FAIR Chef repository catalog containing one or more normalized repository records

## Main steps in this notebook
1. **Define the FAIR Chef repository schema**  
   Create the JSON schema that specifies the structure of repository catalog entries, including the five new raw-friendly fields.

2. **Save the schema**  
   Save the schema as `repository_schema.json`.

3. **Create an empty repository record and catalog container**  
   Create a blank repository record matching the schema structure, and create the repository catalog container that will store multiple repository entries.

4. **Set the repository ID and API URL**  
   Define the re3data repository ID and construct the API endpoint.

5. **Request the XML from the API**  
   Download the repository XML record from re3data.

6. **Parse the XML**  
   Convert the raw XML into a Python XML tree.

7. **Define helper functions**  
   Create reusable helper functions for safely extracting XML values.

8. **Fill basic single-value fields**  
   Extract fields such as ID, name, URL, description, contact, size, and other single-value fields.

9. **Fill simple list fields**  
   Extract repeated text fields such as alternate names, keywords, languages, repository types, and identifiers.

10. **Fill structured fields**  
    Extract nested fields such as subjects, institutions, policies, access, licenses, APIs, and metadata standards.

11. **Fill provenance fields**  
    Add source information, identifiers, dates, and raw XML.

12. **Fill the five new raw-friendly fields**  
    Populate `domains`, `data_types`, `access_profile`, `supported_funders`, and `human_subjects_support` using raw or near-raw values already mapped from the source.

13. **Validate the final repository record**  
    Check that the mapped record matches the JSON schema.

14. **Append the record to the repository catalog**  
    Add the validated repository record to the catalog container.

15. **Save the final repository catalog**  
    Save the final catalog as `repository_catalog.json`.

## Why this matters
This process creates a **normalized, validated, and reusable repository catalog** that FAIR Chef can later search and retrieve to support better repository recommendations for users.  
Instead of relying on unstructured web content, the system can use trusted, structured repository metadata as context for downstream logic and LLM-based guidance.

The added five fields make the catalog more useful for future filtering and matching while still preserving the raw source evidence needed for transparency and later refinement. ```

In [70]:
#Import libraries
import requests
import xml.etree.ElementTree as ET
import json
from jsonschema import validate

In [71]:
# Step 1 — define the schema
repository_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "FAIR Chef Repository Catalog Entry",
    "type": "object",
    "required": ["id", "name", "url", "source"],
    "properties": {
        "id": {"type": "string"},
        "name": {"type": "string"},
        "alternate_names": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "url": {"type": "string"},
        "description": {"type": ["string", "null"]},
        "contact": {"type": ["string", "null"]},
        "repository_types": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "size": {"type": ["string", "null"]},
        "start_date": {"type": ["string", "null"]},
        "languages": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "subjects": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["value"],
                "properties": {
                    "value": {"type": "string"},
                    "scheme": {"type": ["string", "null"]}
                },
                "additionalProperties": False
            },
            "default": []
        },
        "mission_statement_url": {"type": ["string", "null"]},
        "content_types": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "provider_types": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "keywords": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "institutions": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["name"],
                "properties": {
                    "name": {"type": "string"},
                    "alternate_name": {"type": ["string", "null"]},
                    "country": {"type": ["string", "null"]},
                    "responsibilities": {
                        "type": "array",
                        "items": {"type": "string"},
                        "default": []
                    },
                    "institution_type": {"type": ["string", "null"]},
                    "url": {"type": ["string", "null"]},
                    "identifiers": {
                        "type": "array",
                        "items": {"type": "string"},
                        "default": []
                    }
                },
                "additionalProperties": False
            },
            "default": []
        },
        "policies": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": ["string", "null"]},
                    "url": {"type": ["string", "null"]}
                },
                "additionalProperties": False
            },
            "default": []
        },
        "access": {
            "type": "object",
            "properties": {
                "database_access": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                },
                "data_access": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                },
                "data_upload": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                },
                "upload_restrictions": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                }
            },
            "additionalProperties": False,
            "default": {}
        },
        "licenses": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": ["string", "null"]},
                    "url": {"type": ["string", "null"]}
                },
                "additionalProperties": False
            },
            "default": []
        },
        "software": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "versioning": {"type": ["boolean", "null"]},
        "apis": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": ["string", "null"]},
                    "url": {"type": ["string", "null"]}
                },
                "additionalProperties": False
            },
            "default": []
        },
        "persistent_identifier_systems": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "citation_guideline_url": {"type": ["string", "null"]},
        "author_identifier_systems": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "enhanced_publication": {"type": ["boolean", "null"]},
        "quality_management": {"type": ["string", "null"]},
        "metadata_standards": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": ["string", "null"]},
                    "url": {"type": ["string", "null"]},
                    "scheme": {"type": ["string", "null"]}
                },
                "additionalProperties": False
            },
            "default": []
        },
        "remarks": {"type": ["string", "null"]},
        "identifiers": {
            "type": "object",
            "properties": {
                "re3data_id": {"type": ["string", "null"]},
                "repository_identifiers": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                }
            },
            "additionalProperties": False,
            "default": {}
        },
        "dates": {
            "type": "object",
            "properties": {
                "entry_date": {"type": ["string", "null"]},
                "last_update": {"type": ["string", "null"]}
            },
            "additionalProperties": False,
            "default": {}
        },
        "source": {
            "type": "object",
            "required": ["registry"],
            "properties": {
                "registry": {"type": "string"},
                "xml_schema_version": {"type": ["string", "null"]},
                "record_page": {"type": ["string", "null"]},
                "api_endpoint": {"type": ["string", "null"]}
            },
            "additionalProperties": False
        },
        "domains": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["value"],
                "properties": {
                    "value": {"type": "string"},
                    "scheme": {"type": ["string", "null"]}
                },
                "additionalProperties": False
            },
            "default": []
        },
        "data_types": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "access_profile": {
            "type": "object",
            "properties": {
                "database_access": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                },
                "data_access": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                },
                "data_upload": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                },
                "upload_restrictions": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                }
            },
            "additionalProperties": False,
            "default": {}
        },
        "supported_funders": {
            "type": "array",
            "items": {"type": "string"},
            "default": []
        },
        "human_subjects_support": {
            "type": "object",
            "properties": {
                "subject_evidence": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                },
                "description_evidence": {"type": ["string", "null"]},
                "policy_evidence": {
                    "type": "array",
                    "items": {"type": "string"},
                    "default": []
                }
            },
            "additionalProperties": False,
            "default": {}
        },
        "raw_source_record": {"type": ["object", "null"]},
        "custom": {"type": "object", "default": {}}
    },
    "additionalProperties": False
}

print("Schema defined.")

Schema defined.


In [72]:
# Step 2 — save the schema
with open("repository_schema.json", "w", encoding="utf-8") as f:
    json.dump(repository_schema, f, indent=2, ensure_ascii=False)

print("Saved: repository_schema.json")

Saved: repository_schema.json


In [73]:

# Step 3: create an empty repository record and the catalog container
repository_record = {
    "id": None,
    "name": None,
    "alternate_names": [],
    "url": None,
    "description": None,
    "contact": None,
    "repository_types": [],
    "size": None,
    "start_date": None,
    "languages": [],
    "subjects": [],
    "mission_statement_url": None,
    "content_types": [],
    "provider_types": [],
    "keywords": [],
    "institutions": [],
    "policies": [],
    "access": {
        "database_access": [],
        "data_access": [],
        "data_upload": [],
        "upload_restrictions": []
    },
    "licenses": [],
    "software": [],
    "versioning": None,
    "apis": [],
    "persistent_identifier_systems": [],
    "citation_guideline_url": None,
    "author_identifier_systems": [],
    "enhanced_publication": None,
    "quality_management": None,
    "metadata_standards": [],
    "remarks": None,
    "identifiers": {
        "re3data_id": None,
        "repository_identifiers": []
    },
    "dates": {
        "entry_date": None,
        "last_update": None
    },
    "source": {
        "registry": "re3data",
        "xml_schema_version": None,
        "record_page": None,
        "api_endpoint": None
    },
    "domains": [],
    "data_types": [],
    "access_profile": {
        "database_access": [],
        "data_access": [],
        "data_upload": [],
        "upload_restrictions": []
    },
    "supported_funders": [],
    "human_subjects_support": {
        "subject_evidence": [],
        "description_evidence": None,
        "policy_evidence": []
    },
    "raw_source_record": None
    
}

repository_catalog = {
    "catalog_name": "repository_catalog",
    "source": "re3data",
    "repositories": []
}

print("Empty repository record created.")
print("Empty catalog created.")

Empty repository record created.
Empty catalog created.


In [74]:
#step 4: set the repository ID and API URL
repo_id = "r3d100010924"
api_url = f"https://www.re3data.org/api/v1/repository/{repo_id}"

print("Repository ID:", repo_id)
print("API URL:", api_url)

Repository ID: r3d100010924
API URL: https://www.re3data.org/api/v1/repository/r3d100010924


In [75]:
# Step 5 — request the XML from the API
import requests

response = requests.get(api_url, timeout=30)
response.raise_for_status()

print("Request successful.")
print("Status code:", response.status_code)
print(response.text[:1000])

Request successful.
Status code: 200
<?xml version="1.0" encoding="utf-8"?>
<!--re3data.org Schema for the Description of Research Data Repositories. Version 2.2, December 2014. doi:10.2312/re3.006-->
<r3d:re3data xmlns:r3d="http://www.re3data.org/schema/2-2" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.re3data.org/schema/2-2 http://schema.re3data.org/2-2/re3dataV2-2.xsd">
    <r3d:repository>
        <r3d:re3data.orgIdentifier>r3d100010924</r3d:re3data.orgIdentifier>
        <r3d:repositoryName language="eng">OpenNeuro</r3d:repositoryName>
                    <r3d:additionalName language="eng">formerly: OpenfMRI</r3d:additionalName>
                <r3d:repositoryURL>https://openneuro.org/</r3d:repositoryURL>
                    <r3d:repositoryIdentifier>10.25504/FAIRsharing.s1r9bw</r3d:repositoryIdentifier>
                    <r3d:repositoryIdentifier>SCR_005031</r3d:repositoryIdentifier>
                    <r3d:repositoryIdentifier>nlx_14404

In [76]:
# Step 6 — parse the XML
import xml.etree.ElementTree as ET

raw_xml = response.text
root = ET.fromstring(response.content)

print("XML parsed successfully.")
print("Root tag:", root.tag)

XML parsed successfully.
Root tag: {http://www.re3data.org/schema/2-2}re3data


In [77]:
# Step 7 — define helper functions
def get_text(element, path):
    found = element.find(path)
    if found is not None and found.text:
        value = found.text.strip()
        return value if value else None
    return None

def get_all_texts(element, path):
    values = []
    for item in element.findall(path):
        if item.text and item.text.strip():
            values.append(item.text.strip())
    return values

print("Helper functions ready.")

Helper functions ready.


In [78]:
# Step 8 — fill basic single-value fields
repository_record["id"] = get_text(root, ".//{*}re3data.orgIdentifier")
repository_record["name"] = get_text(root, ".//{*}repositoryName")
repository_record["url"] = get_text(root, ".//{*}repositoryURL")
repository_record["description"] = get_text(root, ".//{*}description")
repository_record["contact"] = get_text(root, ".//{*}repositoryContact")
repository_record["size"] = get_text(root, ".//{*}size")
repository_record["start_date"] = get_text(root, ".//{*}startDate")
repository_record["mission_statement_url"] = get_text(root, ".//{*}missionStatementURL")
repository_record["citation_guideline_url"] = get_text(root, ".//{*}citationGuidelineURL")
repository_record["remarks"] = get_text(root, ".//{*}remarks")

print("Basic fields filled.")
print("ID:", repository_record["id"])
print("Name:", repository_record["name"])
print("URL:", repository_record["url"])

Basic fields filled.
ID: r3d100010924
Name: OpenNeuro
URL: https://openneuro.org/


In [79]:
# Step 9 — fill simple list fields
repository_record["alternate_names"] = get_all_texts(root, ".//{*}additionalName")
repository_record["repository_types"] = get_all_texts(root, ".//{*}type")
repository_record["languages"] = get_all_texts(root, ".//{*}repositoryLanguage")
repository_record["content_types"] = get_all_texts(root, ".//{*}contentType")
repository_record["provider_types"] = get_all_texts(root, ".//{*}providerType")
repository_record["keywords"] = get_all_texts(root, ".//{*}keyword")
repository_record["software"] = get_all_texts(root, ".//{*}softwareName")
repository_record["persistent_identifier_systems"] = get_all_texts(root, ".//{*}pidSystem")
repository_record["author_identifier_systems"] = get_all_texts(root, ".//{*}aidSystem")
repository_record["identifiers"]["repository_identifiers"] = get_all_texts(root, ".//{*}repositoryIdentifier")

print("Simple list fields filled.")

Simple list fields filled.


In [80]:
# Step 10 — fill subjects
repository_record["subjects"] = []

for subj in root.findall(".//{*}subject"):
    repository_record["subjects"].append({
        "value": subj.text.strip() if subj.text else None,
        "scheme": subj.attrib.get("subjectScheme")
    })

print("Subjects filled.")
print(repository_record["subjects"][:3])

Subjects filled.
[{'value': '1 Humanities and Social Sciences', 'scheme': 'DFG'}, {'value': '110 Psychology', 'scheme': 'DFG'}, {'value': '12 Social and Behavioural Sciences', 'scheme': 'DFG'}]


In [81]:
# Step 11 — fill institutions
repository_record["institutions"] = []

for inst in root.findall(".//{*}institution"):
    institution_obj = {
        "name": get_text(inst, ".//{*}institutionName"),
        "alternate_name": get_text(inst, ".//{*}institutionAdditionalName"),
        "country": get_text(inst, ".//{*}institutionCountry"),
        "responsibilities": get_all_texts(inst, ".//{*}responsibilityType"),
        "institution_type": get_text(inst, ".//{*}institutionType"),
        "url": get_text(inst, ".//{*}institutionURL"),
        "identifiers": get_all_texts(inst, ".//{*}institutionIdentifier")
    }
    repository_record["institutions"].append(institution_obj)

print("Institutions filled.")
print(repository_record["institutions"][0])

Institutions filled.
{'name': 'National Science Foundation', 'alternate_name': 'NSF', 'country': 'USA', 'responsibilities': ['funding'], 'institution_type': 'non-profit', 'url': 'https://www.nsf.gov/', 'identifiers': ['ROR:021nxhr62', 'RRID:SCR_012938']}


In [82]:
# Step 12: fill policies
repository_record["policies"] = []

for p in root.findall(".//{*}policy"):
    policy_obj = {
        "name": get_text(p, ".//{*}policyName"),
        "url": get_text(p, ".//{*}policyURL")
    }
    repository_record["policies"].append(policy_obj)

print("Policies filled.")
print(repository_record["policies"])

Policies filled.
[{'name': 'OpenNeuro FAQ', 'url': 'https://docs.openneuro.org/faq'}]


In [83]:
# Step 13 — fill access
repository_record["access"]["database_access"] = get_all_texts(root, ".//{*}databaseAccessType")
repository_record["access"]["data_access"] = get_all_texts(root, ".//{*}dataAccessType")
repository_record["access"]["data_upload"] = get_all_texts(root, ".//{*}dataUploadType")
repository_record["access"]["upload_restrictions"] = get_all_texts(root, ".//{*}dataUploadRestriction")

print("Access filled.")
print(repository_record["access"])

Access filled.
{'database_access': ['open'], 'data_access': ['embargoed', 'open', 'restricted'], 'data_upload': ['restricted'], 'upload_restrictions': ['registration']}


In [84]:
# Step 14 — fill licenses
repository_record["licenses"] = []

for lic in root.findall(".//{*}dataLicense"):
    license_obj = {
        "name": get_text(lic, ".//{*}dataLicenseName"),
        "url": get_text(lic, ".//{*}dataLicenseURL")
    }
    repository_record["licenses"].append(license_obj)

print("Licenses filled.")
print(repository_record["licenses"])

Licenses filled.
[{'name': 'CC', 'url': 'https://creativecommons.org/licenses/by/3.0/de/'}, {'name': 'CC0', 'url': 'https://creativecommons.org/publicdomain/zero/1.0/'}, {'name': 'Copyrights', 'url': 'https://docs.openneuro.org/faq'}, {'name': 'ODC', 'url': 'https://www.opendatacommons.org/licenses/pddl/1.0/'}]


In [85]:
# Step 15 — fill versioning, enhanced_publication, and quality_management
versioning_text = get_text(root, ".//{*}versioning")
if versioning_text is None:
    repository_record["versioning"] = None
else:
    repository_record["versioning"] = versioning_text.strip().lower() == "yes"

enhanced_pub_text = get_text(root, ".//{*}enhancedPublication")
if enhanced_pub_text is None:
    repository_record["enhanced_publication"] = None
else:
    repository_record["enhanced_publication"] = enhanced_pub_text.strip().lower() == "yes"

repository_record["quality_management"] = get_text(root, ".//{*}qualityManagement")

print("Special boolean/text fields filled.")

Special boolean/text fields filled.


In [86]:
# Step 16 — fill apis
repository_record["apis"] = []

for api in root.findall(".//{*}api"):
    api_obj = {
        "type": api.attrib.get("apiType"),
        "url": api.text.strip() if api.text else None
    }
    repository_record["apis"].append(api_obj)

print("APIs filled.")
print(repository_record["apis"])

APIs filled.
[{'type': 'REST', 'url': 'https://openneuro.org/crn/graphql'}]


In [87]:
# Step 17 — fill metadata_standards
repository_record["metadata_standards"] = []

for ms in root.findall(".//{*}metadataStandard"):
    metadata_obj = {
        "name": get_text(ms, ".//{*}metadataStandardName"),
        "url": get_text(ms, ".//{*}metadataStandardURL"),
        "scheme": None
    }

    name_elem = ms.find(".//{*}metadataStandardName")
    if name_elem is not None:
        metadata_obj["scheme"] = name_elem.attrib.get("metadataStandardScheme")

    repository_record["metadata_standards"].append(metadata_obj)

print("Metadata standards filled.")
print(repository_record["metadata_standards"])

Metadata standards filled.
[{'name': 'ABCD - Access to Biological Collection Data', 'url': 'http://www.dcc.ac.uk/resources/metadata-standards/abcd-access-biological-collection-data', 'scheme': 'DCC'}]


In [88]:
# Step 18 — fill dates, identifiers.re3data_id, source, and raw XML
repository_record["identifiers"]["re3data_id"] = get_text(root, ".//{*}re3data.orgIdentifier")

repository_record["dates"]["entry_date"] = get_text(root, ".//{*}entryDate")
repository_record["dates"]["last_update"] = get_text(root, ".//{*}lastUpdate")

repository_record["source"]["registry"] = "re3data"
repository_record["source"]["xml_schema_version"] = "2.2"
repository_record["source"]["record_page"] = f"https://www.re3data.org/repository/{repo_id}"
repository_record["source"]["api_endpoint"] = api_url

repository_record["raw_source_record"] = {
    "xml": raw_xml
}

print("Identifiers, dates, source, and raw XML filled.")

Identifiers, dates, source, and raw XML filled.


In [89]:
# Step 19 — fill 5 raw-friendly fields 
# domains: copy from subjects
repository_record["domains"] = [
    {
        "value": s["value"],
        "scheme": s.get("scheme")
    }
    for s in repository_record["subjects"]
    if s.get("value")
]

# data_types: copy directly from content_types
repository_record["data_types"] = repository_record["content_types"][:]

# access_profile: copy directly from access
repository_record["access_profile"] = {
    "database_access": repository_record["access"]["database_access"][:],
    "data_access": repository_record["access"]["data_access"][:],
    "data_upload": repository_record["access"]["data_upload"][:],
    "upload_restrictions": repository_record["access"]["upload_restrictions"][:]
}

# supported_funders: raw names from funding institutions
supported_funders = []
for inst in repository_record["institutions"]:
    if "funding" in inst.get("responsibilities", []):
        if inst.get("alternate_name"):
            supported_funders.append(inst["alternate_name"])
        elif inst.get("name"):
            supported_funders.append(inst["name"])

repository_record["supported_funders"] = sorted(set(supported_funders))

# human_subjects_support: raw evidence only
repository_record["human_subjects_support"] = {
    "subject_evidence": [
        s["value"] for s in repository_record["subjects"] if s.get("value")
    ],
    "description_evidence": repository_record["description"],
    "policy_evidence": [
        p["name"] for p in repository_record["policies"] if p.get("name")
    ]
}

print("Raw-friendly fields filled.")

Raw-friendly fields filled.


In [90]:
# Step 20 — append to the catalog and save the catalog
from jsonschema import validate

validate(instance=repository_record, schema=repository_schema)

repository_catalog["repositories"].append(repository_record)

with open("repository_catalog.json", "w", encoding="utf-8") as f:
    json.dump(repository_catalog, f, indent=2, ensure_ascii=False)

print("Validation passed.")
print("Saved: repository_catalog.json")
print("Number of repositories:", len(repository_catalog["repositories"]))

Validation passed.
Saved: repository_catalog.json
Number of repositories: 1
